# Moontower API Explorer

Scratchpad for hitting every endpoint on `https://api.moontower.ai`.

**Setup** (one-time):
1. `cp .env.example .env` and paste your API key
2. `pip install -r requirements.txt`
3. Run the setup cell below

Each endpoint has its own section — all self-contained, jump to whatever you want.

**API reference:**
- Base: `https://api.moontower.ai/v1`
- Auth: `X-API-Key` header
- Limits: 100 tickers, 31-day range, 30s timeout, 1000 req/min
- Dates: `YYYY-MM-DD`; priority = `trade_date` > `start_date`/`end_date` > latest
- Multi-ticker: repeat the param (`?ticker=SPY&ticker=QQQ`)

## Setup

In [ ]:
from datetime import date, timedelta
import pandas as pd
from moontower import call, to_df

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

TICKERS = ["SPY"]
TODAY = date.today().isoformat()
MONTH_AGO = (date.today() - timedelta(days=30)).isoformat()
YESTERDAY = (date.today() - timedelta(days=1)).isoformat()

print(f"Ready. TICKERS={TICKERS}  MONTH_AGO={MONTH_AGO}  TODAY={TODAY}")

## Reference Data

### `/v1/tickers`

Directory of every ticker the API covers. Run this first to see what's available — the result includes symbol, name, asset class, and any grouping tags. Good sanity check that your key works.

In [ ]:
tickers_df = to_df(call("tickers"))
print(f"{len(tickers_df)} tickers")
tickers_df.head(20)

## Market Data

### `/v1/price`

OHLCV plus mid. Intraday is 15-min delayed. Accepts a single ticker, multiple tickers (repeat the param), a date range via `start_date`/`end_date`, or a single `trade_date`.

In [ ]:
to_df(call("price", ticker=["SPY", "QQQ", "IWM"]))

In [ ]:
to_df(call("price", ticker="SPY", start_date=MONTH_AGO, end_date=TODAY)).tail()

In [ ]:
to_df(call("price", ticker="SPY", trade_date="2025-12-15"))

### `/v1/impliedvol`

Full implied-vol surface by delta level for every listed expiry. Narrow to a specific expiry with `expiry_date` — otherwise you'll get every expiry on that trade date.

In [ ]:
iv = to_df(call("impliedvol", ticker="SPY", trade_date=MONTH_AGO))
iv.head()

In [ ]:
to_df(call("impliedvol", ticker="SPY", trade_date=MONTH_AGO, expiry_date="2026-06-19"))

### `/v1/realvol`

Realized volatility at multiple windows (10d/20d/30d/etc).

**⚠️ EOD-only.** Intraday calls return empty — use `trade_date=<yesterday>` or an explicit end-of-day date.

In [ ]:
to_df(call("realvol", ticker="SPY", trade_date=YESTERDAY))

### `/v1/cmiv`

Constant-maturity implied vol at fixed tenors (30d, 60d, 90d, ...). Use this when you want a clean IV time series that isn't polluted by rolling expiries.

In [ ]:
to_df(call("cmiv", ticker="SPY", start_date=MONTH_AGO, end_date=TODAY)).tail()

### `/v1/ivrank`

IV rank and IV percentile over 1-month, 3-month, and 1-year lookbacks. Fast screener for rich/cheap vol.

In [ ]:
to_df(call("ivrank", ticker=["SPY", "QQQ", "IWM", "TLT"]))

### `/v1/rviv`

Realized vol vs implied vol, with the variance risk premium.

VRP = 100 · ln(IV₃₀ / RV₃₀)

Positive VRP = IV above subsequent RV (the usual state for equity indices).

In [ ]:
to_df(call("rviv", ticker="SPY", start_date=MONTH_AGO, end_date=TODAY)).tail()

### `/v1/skew`

10-delta and 25-delta call/put skew plus historical percentiles. Useful for spotting stretched tails.

In [ ]:
to_df(call("skew", ticker="SPY", trade_date=MONTH_AGO)).head()

### `/v1/optionchain`

Full option chain — every strike, every expiry, IV + greeks.

**⚠️ EOD-only** and **heavy**. Always filter by `expiry_date` unless you really want the whole chain. Use `trade_date` for a past snapshot.

In [ ]:
chain = to_df(call("optionchain", ticker="SPY", trade_date=MONTH_AGO, expiry_date="2026-06-19"))
print(chain.shape)
chain.head()

### `/v1/cockpit`

Bundle: price, IV, returns, RVIV stats in a single response. Handy for dashboards — avoids N round-trips.

In [ ]:
cockpit = to_df(call("cockpit", ticker=["SPY", "QQQ"], start_date=MONTH_AGO, end_date=TODAY))
print(cockpit.shape)
cockpit.tail()

## Trade Ideas

### `/v1/trade-ideas`

Pre-built trade screens. **Gotcha:** `ideas`, `categories`, and `liquidity_levels` are **comma-separated strings** on this endpoint, not repeated params like `ticker`. Easy to get wrong.

In [ ]:
df = to_df(call("trade-ideas"))
print(df.shape)
df.head()

In [ ]:
to_df(call("trade-ideas",
             ticker=["SPY", "QQQ", "IWM", "TLT", "GLD", "USO"],
             liquidity_levels="High,Medium",
             exclude_earnings_weeks=2)).head()

## Bonus: CSV responses

Most endpoints accept `format=csv`. Use `raw=True` on `call()` to get the raw `Response`, then parse with pandas.

In [ ]:
from io import StringIO
resp = call("price", ticker="SPY", start_date=MONTH_AGO, end_date=TODAY, format="csv", raw=True)
pd.read_csv(StringIO(resp.text)).tail()

## Debugging

- **401 / 403** — check `.env` has `MOONTOWER_API_KEY` set and is loaded
- **400** — usually bad date format (`YYYY-MM-DD`) or conflicting date params
- **429** — rate limited; check `X-RateLimit-*` headers below
- **Empty `data`** on intraday `realvol`/`optionchain` — they're EOD-only
- **Unexpected shape on `trade-ideas`** — remember its list params are comma-separated strings

Rate-limit headers on any response:

In [ ]:
resp = call("price", ticker="SPY", raw=True)
{k: v for k, v in resp.headers.items() if "ratelimit" in k.lower()}